In [5]:
import os
from dotenv import load_dotenv

load_dotenv("config.env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY IS NOT FOUND")
print("Gemini API key is loaded")

Gemini API key is loaded


In [6]:
import os
import re 
import json
import numpy as np
import pandas as pd
import glob
import fitz
import faiss
import pickle
from typing import List,Dict, Any
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [7]:
PDF_PATH="data/pdfs"
pdffiles= glob.glob(os.path.join(PDF_PATH, "**","*.pdf"),recursive=True)
print("Number of PDF files found:",len(pdffiles))
for pdf in pdffiles:
    print(pdf)

Number of PDF files found: 2
data/pdfs\Researchpaper1.pdf
data/pdfs\Researchpaper2.pdf


In [8]:
def ext_txt(pdf_path):
    document=fitz.open(pdf_path)
    pages_txt=[]
    for page in document:
        pages_txt.append(page.get_text())
        document.close()
        return "\n".join(pages_txt)

In [9]:
rawpapers=[]
for pdf_path in pdffiles:
    text=ext_txt(pdf_path)
    rawpapers.append({"file_name": os.path.basename(pdf_path),"file_path":pdf_path, "raw_text":text})
papersdf=pd.DataFrame(rawpapers)
print("Papers Loaded:",len(papersdf))

Papers Loaded: 2


In [10]:
for _, row in papersdf.iterrows():
    print("="* 70)
    print("File:", row["file_name"])
    print("Characters:",len(row["raw_text"]))
    print(row["raw_text"][:500])
    print()

File: Researchpaper1.pdf
Characters: 2908
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

File: Researchpaper2.pdf
Characters: 4062
Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin
Ming-Wei Chang
Kenton Lee
Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion

In [11]:
def clean(text):
    text=re.sub(r"\s+", " ", text)
    text=text.strip()
    return text

In [12]:
papersdf["clean"]=papersdf["raw_text"].apply(clean)
print("Text preprocessing done")

Text preprocessing done


In [13]:
for _,row in papersdf.iterrows():
    print("="*70)
    print("File:",row["file_name"])
    print("Cleaned text:")
    print(row["clean"][:1000])
    print()

File: Researchpaper1.pdf
Cleaned text:
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring signi

In [14]:
def ext_metadata(file_name):
    file_name=str(file_name).lower()
    if "researchpaper1" in file_name:
        return{
            "title":"Research Paper 1",
            "publication_year":None,
            "keywords":"research,artificial intelligence, natural language processing"}
    elif "researchpaper2" in file_name:
        return {
            "title":"Research Paper 1",
            "publication_year":None,
            "keywords":"research,artificial intelligence, natural language processing"
        }
    return {
        "title": file_name,
        "publication_year":None,
        "keywords":""
    }


In [15]:
metadata=papersdf["file_name"].apply(ext_metadata)

In [16]:
metadata_df= pd.DataFrame(metadata.tolist())
papersdf=pd.concat([papersdf,metadata_df],axis=1)

In [17]:
papersdf[["file_name","title","keywords"]]

,file_name,title,keywords
0,Researchpaper1.pdf,Research Paper 1,"research,artificial intelligence, natural lang..."
1,Researchpaper2.pdf,Research Paper 1,"research,artificial intelligence, natural lang..."


In [18]:
def split_text(text,chunk_size=1500,chunk_overlap=200):
    chunks=[]
    start=0
    while start< len(text):
        end= start +chunk_size
        chunks.append(text[start:end])
        start+= chunk_size -chunk_overlap
    return chunks



In [19]:

chunks=[]
for _,row in papersdf.iterrows():
    paper_chunks=split_text(row["clean"])
    for i,chunk in enumerate(paper_chunks):
        chunks.append({
            "chunk_id":f"{row['file_name']}_chunk_{i}",
            "file_name":row["file_name"],
            "title":row["title"],
            "publication_year":row["publication_year"],
            "keywords":row["keywords"],
            "text":chunk
        })
chunk_df= pd.DataFrame(chunks)
print("Total chunks:",len(chunk_df))
chunk_df.head()

Total chunks: 7


,chunk_id,file_name,title,publication_year,keywords,text
0,Researchpaper1.pdf_chunk_0,Researchpaper1.pdf,Research Paper 1,NaN,"research,artificial intelligence, natural lang...",Attention Is All You Need Ashish Vaswani∗ Goog...
1,Researchpaper1.pdf_chunk_1,Researchpaper1.pdf,Research Paper 1,NaN,"research,artificial intelligence, natural lang...","5 days on eight GPUs, a small fraction of the ..."
2,Researchpaper1.pdf_chunk_2,Researchpaper1.pdf,Research Paper 1,NaN,"research,artificial intelligence, natural lang...","s of and implementing tensor2tensor, replacing..."
3,Researchpaper2.pdf_chunk_0,Researchpaper2.pdf,Research Paper 1,NaN,"research,artificial intelligence, natural lang...","Proceedings of NAACL-HLT 2019, pages 4171–4186..."
4,Researchpaper2.pdf_chunk_1,Researchpaper2.pdf,Research Paper 1,NaN,"research,artificial intelligence, natural lang...","t), SQuAD v1.1 question answer- ing Test F1 to..."


In [20]:
import os
print("API KEY LOADED:",bool(os.getenv("GOOGLE_API_KEY")))

API KEY LOADED: True


In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [22]:
embeddings= GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [23]:
texts=chunk_df["text"].tolist()
print("Creating embeddings for",len(texts),"chunks...")

Creating embeddings for 7 chunks...


In [25]:
vectors=embeddings.embed_documents(texts)
print("Embeddings created:",len(vectors))
print("Vector dimensions:",len(vectors[0]))

Embeddings created: 7
Vector dimensions: 3072


In [26]:
import chromadb
client= chromadb.Client()
collection=client.get_or_create_collection(name="research_papers")
collection.add(
    ids=chunk_df["chunk_id"].tolist(),
    documents=chunk_df["text"].tolist(),
    embeddings=vectors,
    metadatas=chunk_df[["file_name","title","keywords"]].fillna("").to_dict(orient="records")
)
print("Vector database created successfully")
print("Stored chunks:",collection.count())

Vector database created successfully
Stored chunks: 7


In [ ]:
query="What is the main idea of this research?"
query_vector=embeddings.embed_query(query)
results=collection.query(query_embeddings=[query_vector],n_results=3)
for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:") 
    print(document[:500])

\Resulr 1:
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or
\Resulr 2:
s of and implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating our research. †Work performed while at Google Brain. ‡Work performed while at Google Research. 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
\Resulr 3:
ylor, 1953). The masked language model randomly masks some of the tokens from the input, and the objective is to predict the original vocabulary id of the mas

In [28]:
import google.generativeai as genai
import os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model=genai.GenerativeModel("gemini-3.5-flash-lite")

C:\Users\Dell\AppData\Local\Temp\ipykernel_21612\1155284283.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [31]:
def ask_bot(question):
    query_vector=embeddings.embed_query(question)
    results=collection.query(query_embeddings=[query_vector],n_results=3)
    context="\n\n".join(results["documents"][0])
    prompt=f"""Answer the question using only the context below.
    If the answer is no found,say you cound not find it in the uploaded papers
    Context:{context}
    Question:{question}
    Give a simple and clear answer"""
    response=model.generate_content(prompt)
    return response.text

In [32]:
print(ask_bot("What is the main idea of research paper?"))

Based on the provided context, the main idea of the research paper is to propose a new, simple network architecture called the **Transformer**, which is based solely on attention mechanisms and dispenses with recurrence and convolutions entirely.


In [33]:
question="What is the main idea of research paper1?"
answer=ask_bot(question)
print("Question:",question)
print("Answer:",answer)

Question: What is the main idea of research paper1?
Answer: Based on the provided context, the first research paper ("Attention Is All You Need") proposes a new simple network architecture called the **Transformer**, which is based solely on attention mechanisms and dispenses with recurrence and convolutions entirely.
